# ILSA Survey Analysis - Example Notebook

This notebook demonstrates how to work with the ILSA survey articles dataset.

In [ ]:
import pandas as pd
import json
from pathlib import Path
import sqlite3
import matplotlib.pyplot as plt
import seaborn as sns

# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

## 1. Load JSON Files

In [ ]:
# Load all JSON files
json_dir = Path("ilsa_survey_articles/json")
articles = []

for json_file in json_dir.glob("*.json"):
    with open(json_file) as f:
        articles.append(json.load(f))

print(f"Loaded {len(articles)} articles")

## 2. Convert to DataFrame

In [ ]:
# Create DataFrame
df = pd.json_normalize(articles)
print(f"DataFrame shape: {df.shape}")
print("\nColumns:")
print(df.columns.tolist()[:20])

## 3. Basic Statistics

In [ ]:
# Publication years
year_counts = df['metadata.year'].value_counts().sort_index()
plt.figure(figsize=(10, 5))
year_counts.plot(kind='bar')
plt.title('Publication Years')
plt.xlabel('Year')
plt.ylabel('Number of Articles')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 4. ML Techniques Analysis

In [ ]:
# Extract ML techniques
ml_techniques = []
for article in articles:
    techniques = article['data']['ml_techniques']['all_techniques']
    ml_techniques.extend(techniques)

# Count techniques
technique_counts = pd.Series(ml_techniques).value_counts().head(15)

plt.figure(figsize=(12, 6))
technique_counts.plot(kind='barh')
plt.title('Most Common ML Techniques')
plt.xlabel('Count')
plt.tight_layout()
plt.show()

## 5. Variable Categories

In [ ]:
# Extract variable categories
categories = []
for article in articles:
    for confounder in article['data']['confounders_identified']:
        categories.append(confounder['category'])

category_counts = pd.Series(categories).value_counts()

plt.figure(figsize=(10, 6))
category_counts.plot(kind='pie', autopct='%1.1f%%')
plt.title('Variable Categories Distribution')
plt.ylabel('')
plt.tight_layout()
plt.show()

## 6. SQLite Database

In [ ]:
# Connect to SQLite database
conn = sqlite3.connect("ilsa_survey_articles/ilsa_knowledge_base.db")
cursor = conn.cursor()

# List tables
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
tables = cursor.fetchall()
print("Tables in database:")
for table in tables:
    print(f"  - {table[0]}")

## 7. Parquet File

In [ ]:
# Load parquet file
parquet_df = pd.read_parquet("ilsa_survey_articles/ilsa_master.parquet")
print(f"Parquet DataFrame shape: {parquet_df.shape}")
print("\nFirst few rows:")
print(parquet_df.head())